# Module 3: Exploratory Data Analysis

This notebook implements a complete EDA workflow for the flood prediction dataset. It focuses on understanding the data through statistics and visualization without any preprocessing, feature engineering, or modeling.

## Step 1: Import Libraries

- `numpy`: Numerical computing and array operations.
- `pandas`: Data loading, inspection, and tabular analysis.
- `matplotlib.pyplot`: Low-level plotting and figure creation.
- `seaborn`: High-level statistical visualization.
- `warnings`: Suppress unnecessary warnings during analysis.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils.data_loader import load_dataset
from utils.eda import (
    descriptive_statistics,
    generate_eda_report,
    plot_boxplot,
    plot_distribution,
    plot_heatmap,
    plot_histogram,
    plot_pairplot,
    plot_scatter,
    plot_violin,
    save_figure,
)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

## Step 2: Load Dataset

The dataset is loaded from the raw data folder. The notebook confirms successful loading before continuing.

In [ ]:
project_root = Path.cwd().resolve()
dataset_path = project_root / 'dataset' / 'raw' / 'flood_dataset.xlsx'
df = load_dataset(dataset_path)
print('Dataset loaded successfully.')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('Head:')
display(df.head())
print('Tail:')
display(df.tail())

## Step 3: Dataset Overview

This section summarizes the dataset shape, structure, types, and target variable.

In [ ]:
print('Dataset Shape:', df.shape)
print('Column Names:', list(df.columns))
print('Data Types:')
display(df.dtypes)
print('Target Variable (flood):')
display(df['flood'].value_counts(dropna=False))
print('Numerical Columns:', list(df.select_dtypes(include=[np.number]).columns))
print('Categorical Columns:', list(df.select_dtypes(exclude=[np.number]).columns))
print('Memory Usage (MB):', round(df.memory_usage(deep=True).sum() / (1024 * 1024), 2))

## Step 4: Descriptive Statistical Analysis

Descriptive statistics help quantify central tendency, spread, and overall distribution. Statistics such as mean, median, variance, and skewness are useful for detecting asymmetry and formulating initial observations.

In [ ]:
stats = descriptive_statistics(df)
print('Describe:')
display(stats['describe'])
print('Mean:')
display(stats['mean'])
print('Median:')
display(stats['median'])
print('Mode:')
display(stats['mode'])
print('Variance:')
display(stats['variance'])
print('Standard Deviation:')
display(stats['std'])
print('Minimum:')
display(stats['min'])
print('Maximum:')
display(stats['max'])
print('Skewness:')
display(stats['skewness'])
print('Kurtosis:')
display(stats['kurtosis'])

## Step 5: Target Variable Analysis

The target variable `flood` is analyzed to understand the class distribution. This is important because class balance can influence future model behavior and evaluation.

In [ ]:
target = df['flood']
print('Class Count:')
display(target.value_counts(dropna=False))
print('Class Percentage:')
display(target.value_counts(normalize=True).mul(100).round(2))
print('Class Balance:')
print('Balanced' if target.value_counts().nunique() == 1 else 'Imbalanced')

## Step 6: Univariate Analysis

Each feature is explored individually with histograms, density plots, box plots, violin plots, and count plots where appropriate. These visuals reveal distribution shape, skewness, and potential outliers.

In [ ]:
numeric_features = ['Temp', 'Humidity', 'Cloud Cover', 'ANNUAL', 'Jan-Feb', 'Mar-May', 'Jun-Sep', 'Oct-Dec', 'avgjune']
for feature in numeric_features:
    plot_histogram(df, feature, output_dir=project_root / 'reports' / 'figures')
    plot_distribution(df, feature, output_dir=project_root / 'reports' / 'figures')
    plot_boxplot(df, feature, output_dir=project_root / 'reports' / 'figures')
    plot_violin(df, feature, output_dir=project_root / 'reports' / 'figures')
    print(f'Completed analysis for {feature}')

## Step 7: Distribution Analysis

The rainfall and climate-related variables are reviewed for normal, positively skewed, negatively skewed, or uniform distributions. These patterns help determine possible transformation needs before downstream work.

In [ ]:
for feature in numeric_features:
    print(f'
Feature: {feature}')
    print('Skewness:', round(df[feature].skew(), 3))
    print('Kurtosis:', round(df[feature].kurt(), 3))

## Step 8: Outlier Visualization

Box plots are used only to identify potential outliers. No outlier removal is performed.

In [ ]:
for feature in numeric_features:
    plot_boxplot(df, feature, output_dir=project_root / 'reports' / 'figures')

## Step 9: Multivariate Analysis

Scatter plots and pair plots help reveal relationships among variables, including positive, negative, or weak associations.

In [ ]:
for feature in numeric_features:
    plot_scatter(df, feature, 'flood', output_dir=project_root / 'reports' / 'figures')

plot_pairplot(df, numeric_features + ['flood'], output_dir=project_root / 'reports' / 'figures')

## Step 10: Correlation Analysis

A correlation heatmap highlights strongly related features, moderately related features, and weak connections. This is useful for identifying redundant or informative variables.

In [ ]:
plot_heatmap(df, output_dir=project_root / 'reports' / 'figures')

## Step 11: Feature Relationship with Target

Each feature is compared with the target variable to understand how it may influence flood prediction.

In [ ]:
for feature in numeric_features:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df, x='flood', y=feature, ax=ax)
    ax.set_title(f'{feature} by flood status')
    fig.tight_layout()
    save_figure(fig, f'{feature.lower().replace(  , "_")}_by_flood_boxplot.png', project_root / 'reports' / 'figures')

## Step 12: Insights Report

An automated markdown report is generated to summarize the EDA process and its key observations.

In [ ]:
generate_eda_report(df, output_path=project_root / 'reports' / 'eda_report.md')
print('EDA report generated.')

## Step 13-15: Save Figures, Utility Module, Logging

Figures are written to the reports folder, the reusable EDA utilities are available, and logging captures the main workflow events.